<a href="https://colab.research.google.com/github/atur0021/MMA3001-Project/blob/main/Flight_Data_Extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
import urllib.request

def clean_flight_data(
    input_file: str,
    output_file: str = "cleaned_flight_data.csv",
    sample_interval: float = 0.05,        # 0.05 seconds = 20 Hz
    launch_threshold_g: float = 0.5,      # Acceleration above baseline to detect launch
    g_to_ms2: float = 9.80665             # Standard gravity constant
):
    """
    Cleans flight computer data:
    1. Subtracts 1g stationary baseline offset (calibrates to 0 m/s^2 at rest).
    2. Converts acceleration from g to m/s^2.
    3. Detects launch point (liftoff) and re-indexes time so launch is t = 0.0s.
    4. Resamples data onto a clean 0.05 second grid using linear interpolation.
    """
    # 1. Read input CSV
    try:
        df = pd.read_csv(input_file)
    except Exception as e:
        print(f"Error reading the CSV file from {input_file}.")
        print("Please verify that the URL/path is exactly correct, including casing.")
        raise e

    # Standardize/identify relevant column names (case-insensitive search)
    cols = {c.lower(): c for c in df.columns}

    time_col = next((cols[c] for c in cols if 'time' in c or c == 't'), df.columns[0])
    acc_col  = next((cols[c] for c in cols if 'acc' in c or 'g' in c), df.columns[1])
    agl_col  = next((cols[c] for c in cols if 'agl' in c or 'alt' in c or 'apogee' in c), df.columns[2])

    # Sort chronologically
    df = df.sort_values(by=time_col).reset_index(drop=True)

    # 2. Baseline Gravity Offset & Conversion to m/s^2
    # Estimate resting baseline from the first 50 samples (or first 10% of data)
    pre_launch_window = min(50, len(df) // 10)
    baseline_acc_g = df[acc_col].iloc[:pre_launch_window].mean()

    # Subtract baseline so stationary acceleration is 0 g, then convert to m/s^2
    net_acc_g = df[acc_col] - baseline_acc_g
    acc_ms2 = net_acc_g * g_to_ms2
    df['acc_ms2'] = acc_ms2

    # 3. Detect Launch (Liftoff) Point
    # Identify first index where net acceleration exceeds threshold
    threshold_ms2 = launch_threshold_g * g_to_ms2
    launch_mask = df['acc_ms2'] >= threshold_ms2

    if launch_mask.any():
        launch_idx = launch_mask.idxmax()
    else:
        print("Warning: Launch threshold not exceeded. Defaulting to t=0 at start of file.")
        launch_idx = 0

    launch_time = df[time_col].iloc[launch_idx]

    # Filter data from launch onward and re-zero time
    df_flight = df.iloc[launch_idx:].copy()
    df_flight['t_shifted'] = df_flight[time_col] - launch_time

    # 4. Resample onto exact 0.05 second grid
    t_min = 0.0
    t_max = df_flight['t_shifted'].max()
    resampled_time = np.arange(t_min, t_max + sample_interval, sample_interval)

    # Linear interpolation onto new uniform 0.05s grid
    resampled_acc = np.interp(resampled_time, df_flight['t_shifted'], df_flight['acc_ms2'])
    resampled_agl = np.interp(resampled_time, df_flight['t_shifted'], df_flight[agl_col])

    # 5. Build clean DataFrame and export CSV
    cleaned_df = pd.DataFrame({
        'Time_s': np.round(resampled_time, 3),
        'Acceleration_ms2': np.round(resampled_acc, 4),
        'Altitude_AGL_m': np.round(resampled_agl, 2)
    })

    cleaned_df.to_csv(output_file, index=False)
    print(f"Successfully processed {len(cleaned_df)} rows. Saved to '{output_file}'.")
    return cleaned_df

In [16]:
import os
from google.colab import userdata
import requests
import pandas as pd
import numpy as np

# --- 1. SECURELY DOWNLOAD THE FILE DIRECTLY ---
try:
    token = userdata.get('GH_PAT')
except Exception:
    raise ValueError("Please add your GitHub Personal Access Token to Colab Secrets with the key name 'GH_PAT'.")

owner = "atur0021"
repo = "MMA3001-Project"
branch = "main"
path = "data/Raven4_Meridian_Launch_1.csv"

# Bypassing the API cache by pulling directly from raw.githubusercontent.com with token authentication
raw_url = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{path}"
headers = {"Authorization": f"token {token}"}

print("Downloading flight data directly from raw server...")
response = requests.get(raw_url, headers=headers)

if response.status_code == 200:
    local_filename = "Raven4_Meridian_Launch_1.csv"
    with open(local_filename, "wb") as f:
        f.write(response.content)
    print(f"Successfully downloaded '{local_filename}' ({len(response.content) / (1024*1024):.2f} MB)!")
else:
    raise RuntimeError(f"Download failed with status {response.status_code}: {response.text}")

# --- 2. THE SIMPLIFIED CLEANING FUNCTION WITH UNIT CONVERSION ---
def clean_flight_data(input_file, output_file="cleaned_flight_data.csv", sample_interval=0.05, launch_threshold_g=0.5, g_to_ms2=9.80665, ft_to_m=0.3048):
    df = pd.read_csv(input_file)
    df = df.sort_values(by=df.columns[0]).reset_index(drop=True)

    # Dynamic column identification
    time_col = next((c for c in df.columns if 'time' in c.lower() or c.lower() == 't'), df.columns[0])
    acc_col  = next((c for c in df.columns if 'acc' in c.lower() or 'g' in c.lower()), df.columns[1])
    agl_col  = next((c for c in df.columns if 'agl' in c.lower() or 'alt' in c.lower() or 'apogee' in c.lower()), df.columns[2])

    # Baseline Gravity Offset Adjustment
    pre_launch_window = min(50, len(df) // 10)
    baseline_acc_g = df[acc_col].iloc[:pre_launch_window].mean()
    df['acc_ms2'] = (df[acc_col] - baseline_acc_g) * g_to_ms2

    # Launch Point Alignment
    launch_mask = df['acc_ms2'] >= (launch_threshold_g * g_to_ms2)
    launch_idx = launch_mask.idxmax() if launch_mask.any() else 0
    df_flight = df.iloc[launch_idx:].copy()
    df_flight['t_shifted'] = df_flight[time_col] - df_flight[time_col].iloc[0]

    # Resample (Linear Interpolation)
    resampled_time = np.arange(0.0, df_flight['t_shifted'].max() + sample_interval, sample_interval)
    resampled_acc = np.interp(resampled_time, df_flight['t_shifted'], df_flight['acc_ms2'])

    # Interpolating altitude (converting feet to meters directly during calculation)
    resampled_agl_m = np.interp(resampled_time, df_flight['t_shifted'], df_flight[agl_col]) * ft_to_m

    # Save and display output
    cleaned_df = pd.DataFrame({
        'Time_s': np.round(resampled_time, 3),
        'Acceleration_ms2': np.round(resampled_acc, 4),
        'Altitude_AGL_m': np.round(resampled_agl_m, 2)
    })
    cleaned_df.to_csv(output_file, index=False)
    print(f"Successfully processed and saved {len(cleaned_df)} rows to '{output_file}'.")
    return cleaned_df

# --- 3. RUN PROCESS ---
cleaned_df = clean_flight_data(local_filename)
display(cleaned_df.head())


Successfully downloaded 'Raven4_Meridian_Launch_1.csv' (13.59 MB)!
Successfully processed and saved 5136 rows to 'cleaned_flight_data.csv'.


,Time_s,Acceleration_ms2,Altitude_AGL_m
0,0.00,8.8059,-0.16
1,0.05,98.2108,-0.16
2,0.10,108.7863,-0.16
3,0.15,105.9022,0.85
4,0.20,96.2887,1.53
